In [0]:
-- ============================================
-- BRONZE 层: 增量数据加载 (使用 MERGE)
-- ============================================

-- 1. 创建临时视图连接 MySQL (使用 Databricks 的 MySQL 连接器)
CREATE OR REPLACE TEMPORARY VIEW mysql_user_info_latest AS
SELECT 
    id          ,
    login_name            ,
    nick_name             ,
    passwd                ,
    name                  ,
    phone_num             ,
    email                 ,
    head_img              ,
    user_level            ,
    birthday              ,
    gender                ,
    create_time           ,
    COALESCE(operate_time,create_time) as operate_time          ,
    status                ,
    CURRENT_TIMESTAMP() as _bronze_load_ts,
    'mysql_production' as _source_system,
    'user_info' as _source_table
FROM awsmysql_catalog.gmall.user_info
WHERE COALESCE(operate_time,create_time) > (
    SELECT COALESCE(MAX(COALESCE(operate_time,create_time)), '1900-01-01')
    FROM aws3.bronze.user_info
);
---所以要定期-- 清理超过保留期限的文件（默认清理7天前的）我的表设置一天
--VACUUM your_table_name;实际数据在云上，但对DB，已经没了


-- 2. MERGE 到 Bronze 层
MERGE INTO aws3.bronze.user_info AS target
USING mysql_user_info_latest AS source
ON target.id = source.id 
   AND target._source_system = source._source_system
   AND COALESCE(target.operate_time,target.create_time) = COALESCE(source.operate_time,source.create_time)
WHEN NOT MATCHED THEN
INSERT (
    _bronze_load_ts,
    _bronze_load_id,
    _source_system,
    _source_table,
    id          ,
    login_name            ,
    nick_name             ,
    passwd                ,
    name                  ,
    phone_num             ,
    email                 ,
    head_img              ,
    user_level            ,
    birthday              ,
    gender                ,
    create_time           ,
    operate_time          ,
    status                
)
VALUES (
    source._bronze_load_ts,
    UUID() ,--as _bronze_load_id,
    source._source_system,
    source._source_table,
    source.id          ,
    source.login_name            ,
    source.nick_name             ,
    source.passwd                ,
    source.name                  ,
    source.phone_num             ,
    source.email                 ,
    source.head_img              ,
    source.user_level            ,
    source.birthday              ,
    source.gender                ,
    source.create_time           ,
    source.operate_time          ,
    source.status
);